# Optimisation via GridSearchCV

Pour la regession logisitique:
- penalty - qu'est qui on punit ?
  si un modele s'overfit, ca veut dire qu'il apprend le training data too well et il apprend ce qui est nois; modele veut faire les coefficients (weights) as large as possible to fit the traiing data perfectly.
  L2: ridge- shrinks all weight toward 0 but never full 0; all features stay in model, just with smaller influence
  L1: lasso - pushes some weights all the way to zero; some features get eliminated; useful when we have a lot of useless columns


- les algos qui existent ("solver")
  liblinear : qui fonctionne bien sur les donnees plus petites, support L1 et L2
  saga : qui marche bien sur les donnees plus larges, also supports L1 et L2
  lbfgs : default , fast, but only supports L2 

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import average_precision_score, precision_recall_curve

param_grid = {
    'C': [0.01, 0.1, 1, 10], # control la strictness du modele
    'penalty': ['l1', 'l2'], # adding penalty for complexity 
    'solver': ['liblinear', 'saga'], #solver - how does it actually find the answer - algo that seraches 
}

grid = GridSearchCV(
    LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    param_grid,
    scoring='average_precision', # vu commme meme si nos classes sont balanced, que dans training, et non pas en test
    cv=5, # 5 cross validation folds
    n_jobs=-1, #how many CPU cores to use in parallel
    # -1 use all avaialble cores to make faster
    verbose=1
)

grid.fit(X_train_scaled, y_train)

print("Meilleurs paramètres :", grid.best_params_)
print("Meilleur Average Precision (CV)  :", grid.best_score_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' 

Meilleurs paramètres : {'C': 0.01, 'penalty': 'l2', 'solver': 'saga'}
Meilleur Average Precision (CV)  : 0.6241343652414262


/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/ankim/Desktop/Projet3/.venv/lib/python3.14/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/anki

In [ ]:
results = grid.cv_results_
best_index = grid.best_index_
fold_scores = [results[f'split{i}_test_score'][best_index] for i in range(5)]
cv_mean, cv_std = np.mean(fold_scores), np.std(fold_scores)

# eval le meilleur sur le test set
best_model = grid.best_estimator_
y_pred  = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
print(classification_report(y_test, y_pred))
print("AUROC :", roc_auc_score(y_test, y_proba))
print("AUPRC test:", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.95      0.78      0.85       247
           1       0.40      0.77      0.52        47

    accuracy                           0.78       294
   macro avg       0.67      0.77      0.69       294
weighted avg       0.86      0.78      0.80       294

AUROC : 0.8311654750624515
AUPRC test: 0.5512886979876199


In [ ]:
# Pour ne pas rester avec que le 0.5 threshold ; trouvons un qui est better:

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1])  
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

Best threshold: 0.5286413344157181


In [ ]:
# use that cutoff to turn probabilities into actual yes/no predictions avec y _pred mais demander a Fernanda
y_pred = (y_proba >= best_threshold).astype(int)

In [ ]:
# mettre tous together so this model can be compared to the others later
result = {
    'best_params': grid.best_params_,
    'cv_fold_scores': fold_scores,
    'cv_mean': cv_mean,
    'cv_std': cv_std,
    'test_auroc': roc_auc_score(y_test, y_proba),
    'test_auprc': average_precision_score(y_test, y_proba),
    'threshold': best_threshold,
    'precision': precisions[best_idx],
    'recall': recalls[best_idx],
    'f1': f1_scores[best_idx],
    'precisions_curve': precisions,
    'recalls_curve': recalls,
    'report': classification_report(y_test, y_pred),
}

print(f"CV mean / std (AUPRC): {cv_mean:.3f} / {cv_std:.3f}")
print(f"Test AUROC: {result['test_auroc']:.3f}   Test AUPRC: {result['test_auprc']:.3f}")
print(f"Chosen threshold: {best_threshold:.3f}")
print(f"Precision: {result['precision']:.3f}   Recall: {result['recall']:.3f}   F1: {result['f1']:.3f}")
print(result['report'])

CV mean / std (AUPRC): 0.624 / 0.067
Test AUROC: 0.831   Test AUPRC: 0.551
Chosen threshold: 0.529
Precision: 0.414   Recall: 0.766   F1: 0.537
              precision    recall  f1-score   support

           0       0.95      0.79      0.86       247
           1       0.41      0.77      0.54        47

    accuracy                           0.79       294
   macro avg       0.68      0.78      0.70       294
weighted avg       0.86      0.79      0.81       294



Pour nous, le recall est le plus importante; entre les deux modeles (pas opti et opti), le recall pour les gens qui sont partis a beaucoup ameliore (43% to 79%)!

notes le 19 juin - 
deja metrics d'evaluation - AUROC --> Average Precision 
// structurer le notebook - creer un dataframe - nom du modele(first colonne), les different parametres, metrics (recall , precision, seuile d'evaluation par defaut et des differentes resultats)

point 2: meuilleur seuil - why?
- linear : logitisique
- non linear : forest
- essayer de chercher quels sont modeles linaires et non linaires !! STEP 1

point 3: non optimises -> petite table avec le differentes modeles  THEN optimises 
Quel sont les meuilleurs resultats avec non-opti et opti???




- Class Weighting : tell the nodel to penalize getting the minority class wrong more heavily

In [ ]:
# from sklean.ensemble import RandomForestClassifier

# SOIT:
# model = RandomForestClassifier(class_weight='balanced') 
# SOIT:
# model = RandomForestClassifier(class_weight={0:1, 1:5})

- SMOTE (Synthetic Minority Oversampling Technique) : generer les nouveaux, synthetique minority-class exemples; donc on va de 247 restes/ 27 partis a 247 restes/ 247 partis

In [ ]:
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_resamples, y_resampled = smote.fit_resample(X_train, y_train)

## PRIORITIZE THIS!!!!!!!
##et puis ensuite la split 

## Les paramètres clés par modèle

### Random Forest
- **n_estimators** — nombre d'arbres construits. Plus il y en a, plus le modèle est stable — mais plus c'est lent à entraîner.
- **max_depth** — profondeur maximale de chaque arbre. Sans limite, l'arbre mémorise le training set (overfitting). On le contraint pour généraliser.
- **min_samples_leaf** — nombre minimum d'observations requis dans une feuille. Plus la valeur est grande, plus l'arbre est simple et moins il overfit. (one leaf  means that we can just create branch to classify weird outlier)

- A noter : ici on essaie un autre format de class weight pour balancer nos classes !


In [ ]:
# Pour les otpimisations:
# gradient = + learning_rate
# SVC = C, kernel ,gamma
param_grid = {
    'n_estimators': [100, 200, 300], # combien des arbres a construire
    'max_depth': [3, 5, None], # quel profondeur pour chaque arbre 
    'min_samples_leaf': [3, 5, 10]
} 

grid = GridSearchCV(
    RandomForestClassifier(class_weight={0: 1, 1: 5}, random_state=42),
    param_grid,
    scoring='average_precision',
    cv=5, # 5 cross validation folds
    n_jobs=-1, #ho many CPU cores to use in parallel
    # -1 use all avaialble cores to make faster
    verbose=1
)

grid.fit(X_train_scaled, y_train)

print("Meilleurs paramètres :", grid.best_params_)
print("Meilleur Average Precision (CV)  :", grid.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Meilleurs paramètres : {'max_depth': None, 'min_samples_leaf': 10, 'n_estimators': 200}
Meilleur Average Precision (CV)  : 0.5064471221550911


In [ ]:
# evalue le meilleur modele sur le test set
best_model = grid.best_estimator_
y_pred  = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
print(classification_report(y_test, y_pred))
print("AUROC :", roc_auc_score(y_test, y_proba))
print("AUPRC test:", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.91      0.84      0.88       247
           1       0.41      0.57      0.48        47

    accuracy                           0.80       294
   macro avg       0.66      0.71      0.68       294
weighted avg       0.83      0.80      0.81       294

AUROC : 0.7680248083383582
AUPRC test: 0.3815295743225043


Pour le random forest, on ameliore mais legerement vis-a-vis le recall (etonnante car normalement le Random Forest est bien) - on va de 0.28 a 0.57

### Gradient Boosting
- **max_iter** — nombre d'étapes de correction séquentielles. Pour une classification binaure, chaque iteration ajoute une nouvelle tree; plus des trees, plus capable a capturer les patterns complexe, mais aussi de fitter noise !
- **learning_rate** — taille de chaque correction. Petit = corrections prudentes, modèle plus précis mais nécessite plus d'arbres. Grand = apprentissage rapide mais risque d'overfitting. High rate means you add big weights each time, low rate you add tiny weights each time ; cela est tres important pour Gradient Boosting qui fonctionne par la construction des arbes sequentially, chaque arbre qui corrige les erreurs d'ensemble.
- **max_leaf_nodes** - pour HISTGB; la, en fait, pour chaque etape, on regarde l'abre ENTIER et puis on split sur un leaf ou de spliter va reduire les erreurs le plus
- **max_depth** — profondeur de chaque arbre correcteur. Les arbres de Gradient Boosting sont volontairement petits (3-5), contrairement à Random Forest.

In [ ]:
param_grid = {
    'max_iter': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_leaf_nodes': [15, 31, 63], 
    'max_depth': [2, 3, 4, None],
    'min_samples_leaf': [3, 5, 10, 20],
} 

grid = GridSearchCV(
    HistGradientBoostingClassifier(class_weight='balanced', random_state=42),
    param_grid,
    scoring='average_precision',
    cv=5, # 5 cross validation folds
    n_jobs=-1, #how many CPU cores to use in parallel
    # -1 use all avaialble cores to make faster
    verbose=1
)

grid.fit(X_train_scaled, y_train)

print("Meilleurs paramètres :", grid.best_params_)
print("Meilleur Average Precision (CV)  :", grid.best_score_)

Fitting 5 folds for each of 576 candidates, totalling 2880 fits


KeyboardInterrupt: 

In [ ]:
best_model = grid.best_estimator_
y_pred  = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
print(classification_report(y_test, y_pred))
print("AUROC :", roc_auc_score(y_test, y_proba))
print("AUPRC test:", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.93      0.81      0.86       247
           1       0.39      0.66      0.49        47

    accuracy                           0.78       294
   macro avg       0.66      0.73      0.68       294
weighted avg       0.84      0.78      0.80       294

AUROC : 0.7955034886725816
AUPRC test: 0.4886726718084531


On augment niveau de recall, quasi double 0.36->0.66)
precision: 52% --> 39%

precision-recall curve (PR):  
calcul seuil optimial - permet de ne pas trop degradee precision mais monter recall)
check the curb
optimiser - i have these results, mais il existe des autres solutions - mnt comment on gere les classes imbalanced

### SVC
- **C** — tolérance aux erreurs. Grand C = frontière stricte, essaie de tout classer correctement (risque d'overfitting). Petit C = accepte quelques erreurs pour une frontière plus générale. high - wiggly line, low C - clean line
- **kernel** — forme de la frontière. `rbf` pour des données non-linéaires (cas le plus courant), `linear` si les classes sont déjà bien séparables.
- **gamma** — influence de chaque point sur la frontière. Valeur élevée = chaque point influence peu de voisins (frontière complexe). `scale` est un bon point de départ. each data point influences where boundary line goes - gamma controls how far that influence reaches;
  --> high gamma: each point only influence its immediate neightbors
  --> low gamma- each point influences a wide area around it (boundary is smoother and more general)

In [ ]:
# param_grid = {
#     'C': [0.1, 1, 10, 100],  # how strictly to fit the training data
#     'kernel': ['linear', 'rbf'], # shape of the decision boundary
#     'gamma': ['scale', 'auto', 0.01, 0.1, 1], # only used by 'rbf', ignored for 'linear'
# }

# grid = GridSearchCV(
#     SVC(class_weight={0: 1, 1: 5}, probability=True, random_state=42),
#     param_grid,
#     scoring='average_precision',
#     cv=5,
#     n_jobs=-1,
#     verbose=1
# )
# grid.fit(X_train_scaled, y_train)
# print("Meilleurs paramètres :", grid.best_params_)
# print("Meilleur Average Precision (CV)  :", grid.best_score_)

In [ ]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV

base_svc = SVC(class_weight={0: 1, 1: 5}, random_state=42)
calibrated_svc = CalibratedClassifierCV(base_svc, ensemble=False)

param_grid = {
    'estimator__C': [0.1, 1, 10, 100],
    'estimator__kernel': ['linear', 'rbf'],
    'estimator__gamma': ['scale', 'auto', 0.01, 0.1, 1],
}

grid = GridSearchCV(
    calibrated_svc,
    param_grid,
    scoring='average_precision',
    cv=5,
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train_scaled, y_train)

Fitting 5 folds for each of 40 candidates, totalling 200 fits


KeyboardInterrupt: 

In [ ]:
print("Meilleurs paramètres :", grid.best_params_)
print("Meilleur Average Precision (CV)  :", grid.best_score_)
best_model = grid.best_estimator_
y_pred  = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]
print(classification_report(y_test, y_pred))
print("AUROC :", roc_auc_score(y_test, y_proba))
print("AUPRC test:", average_precision_score(y_test, y_proba))

Meilleurs paramètres : {'estimator__C': 0.1, 'estimator__gamma': 'scale', 'estimator__kernel': 'linear'}
Meilleur Average Precision (CV)  : 0.5744015363913697
              precision    recall  f1-score   support

           0       0.87      0.99      0.92       247
           1       0.77      0.21      0.33        47

    accuracy                           0.86       294
   macro avg       0.82      0.60      0.63       294
weighted avg       0.85      0.86      0.83       294

AUROC : 0.8418468429666639
AUPRC test: 0.5807435240235718


Choquant - on a baisse en niveau de recall de 0.49 a 0.21.

Le meuilleur modele est donc Logistic Regression avec un recall pour la classe 1 a 77%

notes le 19 juin - 
deja metrics d'evaluation - AUROC --> Average Precision 
// structurer le notebook - creer un dataframe - nom du modele(first colonne), les different parametres, metrics (recall , precision, seuile d'evaluation par defaut et des differentes resultats)

point 2: meuilleur seuil - why?
- linear : logitisique
- non linear : forest
- essayer de chercher quels sont modeles linaires et non linaires !! STEP 1

point 3: non optimises -> petite table avec le differentes modeles  THEN optimises 
Quel sont les meuilleurs resultats avec non-opti et opti???




- SMOTE (Synthetic Minority Oversampling Technique) : generer les nouveaux, synthetique minority-class exemples; donc on va de 247 restes/ 27 partis a 247 restes/ 247 partis

- Class Weighting : tell the nodel to penalize getting the minority class wrong more heavily

In [ ]:
# from sklean.ensemble import RandomForestClassifier

# SOIT:
# model = RandomForestClassifier(class_weight='balanced') 
# SOIT:
# model = RandomForestClassifier(class_weight={0:1, 1:5})

In [ ]:
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_resamples, y_resampled = smote.fit_resample(X_train, y_train)

## PRIORITIZE THIS!!!!!!!
##et puis ensuite la split 